# Fetch Zendesk Tickets

This notebook fetches tickets from Zendesk API and saves them to `ticket_data.json`.

## Import Required Libraries

In [1]:
import requests
from requests.auth import HTTPBasicAuth
import json
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

## Configure Zendesk API Credentials

Load Zendesk credentials from environment variables.

In [2]:
# Zendesk API configuration
ZENDESK_DOMAIN = os.getenv("ZENDESK_DOMAIN")
ZENDESK_EMAIL = os.getenv("ZENDESK_EMAIL")
ZENDESK_API_TOKEN = os.getenv("ZENDESK_API_TOKEN")

# Construct the API URL
ZENDESK_URL = f"https://{ZENDESK_DOMAIN}/api/v2/tickets"

# Set up authentication
auth = HTTPBasicAuth(f'{ZENDESK_EMAIL}/token', ZENDESK_API_TOKEN)

print(f"Zendesk Domain: {ZENDESK_DOMAIN}")
print(f"API URL: {ZENDESK_URL}")

Zendesk Domain: self-37540.zendesk.com
API URL: https://self-37540.zendesk.com/api/v2/tickets


In [3]:
# Debug: Check credentials (masking sensitive parts)
print(f"Email: {ZENDESK_EMAIL}")
print(f"Token (first 10 chars): {ZENDESK_API_TOKEN[:10]}...")
print(f"Token length: {len(ZENDESK_API_TOKEN) if ZENDESK_API_TOKEN else 0}")
print(f"Auth string: {ZENDESK_EMAIL}/token")
print(f"\nFull auth object: {auth}")

Email: kitty-katxx@hotmail.com
Token (first 10 chars): BmIPg0HHUg...
Token length: 40
Auth string: kitty-katxx@hotmail.com/token

Full auth object: <requests.auth.HTTPBasicAuth object at 0x10439e910>


## Fetch Tickets from Zendesk API

Retrieve all tickets using the Zendesk API with pagination support.

In [4]:
def fetch_zendesk_tickets():
    """
    Fetch all tickets from Zendesk API with pagination support.
    """
    all_tickets = []
    url = ZENDESK_URL
    
    # Set up headers
    headers = {
        "Content-Type": "application/json",
    }
    
    while url:
        print(f"Fetching tickets from: {url}")
        
        try:
            response = requests.request(
                "GET",
                url,
                auth=auth,
                headers=headers
            )
            response.raise_for_status()
            
            data = response.json()
            tickets = data.get('tickets', [])
            all_tickets.extend(tickets)
            
            print(f"Retrieved {len(tickets)} tickets (Total: {len(all_tickets)})")
            
            # Check for next page
            url = data.get('next_page')
            
        except requests.exceptions.RequestException as e:
            print(f"❌ Error fetching tickets: {e}")
            break
    
    return all_tickets

# Fetch all tickets
print("Starting ticket fetch...\n")
tickets = fetch_zendesk_tickets()
print(f"\n✅ Total tickets fetched: {len(tickets)}")

Starting ticket fetch...

Fetching tickets from: https://self-37540.zendesk.com/api/v2/tickets
Retrieved 16 tickets (Total: 16)

✅ Total tickets fetched: 16
Retrieved 16 tickets (Total: 16)

✅ Total tickets fetched: 16


## Save Tickets to JSON File

Save the fetched tickets to `ticket_data.json`.

In [5]:
# Save tickets to JSON file
output_file = "ticket_data.json"

with open(output_file, 'w') as f:
    json.dump(tickets, f, indent=2)

print(f"✅ Tickets saved to {output_file}")
print(f"File size: {os.path.getsize(output_file)} bytes")

✅ Tickets saved to ticket_data.json
File size: 63068 bytes


## Display Sample Ticket Data

Show a preview of the first ticket.

In [6]:
# Display a sample ticket
if tickets:
    print("Sample ticket data:")
    print(json.dumps(tickets[0], indent=2))
else:
    print("No tickets found.")

Sample ticket data:
{
  "url": "https://self-37540.zendesk.com/api/v2/tickets/1.json",
  "id": 1,
  "external_id": null,
  "via": {
    "channel": "sample_ticket",
    "source": {
      "from": {},
      "to": {},
      "rel": null
    }
  },
  "created_at": "2026-03-03T00:08:08Z",
  "updated_at": "2026-03-03T00:08:08Z",
  "generated_timestamp": 1772496488,
  "type": null,
  "subject": "SAMPLE: How does Zendesk work",
  "raw_subject": "SAMPLE: How does Zendesk work",
  "description": "Hello, let's see how you or your agents can easily respond to and solve tickets.\n\nFeel free to email additional customer test inquiries to **support@self-37540.zendesk.com**.\n\nBut first, let's start by solving one ticket.\n\nYour Zendesk Team",
  "priority": "normal",
  "status": "open",
  "recipient": null,
  "requester_id": 47307805432091,
  "submitter_id": 47307805432091,
  "assignee_id": 47307721291035,
  "organization_id": null,
  "group_id": 47307721895195,
  "collaborator_ids": [
    4730772129